In [1]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm


from glob import glob
import psi4
from helper_CC_ML_spacial import *



  Threads set to 12 by Python driver.


In [2]:
pd.read_csv("results.csv",index_col=0)

,basis set,PySCF_Corr,Psi4_Corr,Abs. Dev.,PySCF_norm_t1,Psi4_norm_t1,PySCF_norm_t2,Psi4_norm_t2
structure,,,,,,,,
HLi,STO-3G,-0.020276,-0.020276,1.026300e-09,0.038929,0.038929,0.152045,0.152045
FF,aug-cc-pVDZ,-0.437193,-0.437193,1.867580e-09,0.051552,0.051552,0.310279,0.310279
FF,STO-3G,-0.078443,-0.078443,3.254190e-09,0.016500,0.016500,0.258069,0.258069
BF,STO-3G,-0.078625,-0.078625,3.914590e-09,0.068253,0.068253,0.246282,0.246282
BH,STO-3G,-0.057412,-0.057412,8.111790e-09,0.022014,0.022014,0.299934,0.299934
...,...,...,...,...,...,...,...,...
FO,aug-cc-pVDZ,-0.384824,-0.549252,1.644284e-01,0.156001,0.515998,0.409078,0.442873
FO,cc-pVDZ,-0.358196,-0.526366,1.681700e-01,0.148749,0.519507,0.381658,0.442259
FO,STO-3G,-0.063092,-0.237578,1.744855e-01,0.138718,0.591451,0.245255,0.485122


# These are the machine learning features, these will be useful later

In [3]:
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

# Use the basis sets for both Psi4 and PySCF

In [4]:

basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

# Load in the xyz coordinates of the system and then run Psi4 with the Psi4Numpy code we use for DDCC

In [5]:

with open('diatomics/NN.xyz','r') as f:
    text=f.read()


qmol = psi4.qcdb.Molecule.from_string(text, dtype='xyz')
mol = psi4.geometry(qmol.create_psi4_string_from_molecule()+ 'symmetry c1')                

psi4.core.clean()
psi4.core.be_quiet()

psi4.set_options({'basis': basis_sets[0],
                  'scf_type':     'pk',
                  'reference':    'rohf',
                  'mp2_type':     'conv',
                  'e_convergence': 1e-8,
                  'd_convergence': 1e-8})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)

A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)

A.compute_energy()



Computing RHF reference.


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 2 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 10 basis functions.
(10, 10)
(8, 8)
Building initial guess...

..initialized CCSD in 0.020 seconds.

CCSD Iteration   0: CCSD correlation = -0.154646624232695   dE =  1.54647E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.147891140590810   dE =  6.75548E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.151291559003040   dE = -3.40042E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.152315840580268   dE = -1.02428E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.153350720965911   dE = -1.03488E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.153514987399625   dE = -1.64266E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.153509819978286   dE =  5.16742E-06   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.153512855321501   dE = -3.03534E-06   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.153512629622595   dE =  

np.float64(-0.1535126969598729)

In [6]:
rhf_e+ A.FinalEnergy

np.float64(-107.65011132047981)

# Correlation energy

In [7]:
A.FinalEnergy

np.float64(-0.1535126969598729)

# T1-amplitudes

In [8]:

A.t1

array([[-2.90048516e-18,  1.19963617e-16,  9.78598609e-16],
       [ 2.19677714e-18,  7.58030184e-17,  1.02340840e-02],
       [ 6.71939591e-16, -8.65924244e-17, -1.12089149e-17],
       [ 4.04881628e-17, -1.81424724e-15, -2.24884049e-16],
       [ 1.13627970e-18, -4.98294534e-16,  1.22206537e-15]])

# T2-amplitudes

In [9]:
A.t2

array([[[[-1.12324579e-02, -1.31046021e-18,  1.02564084e-19],
         [-1.30902526e-18, -1.12324579e-02,  2.05099976e-17],
         [ 1.02564084e-19,  2.05099976e-17, -2.40315812e-02]],

        [[-4.31906320e-16, -3.54659863e-18,  1.53197186e-19],
         [-3.01509430e-18, -4.49494387e-16,  1.39058449e-17],
         [ 2.84431226e-19,  2.53810179e-17,  1.56786560e-16]],

        [[-3.78845131e-19, -2.03571907e-17,  1.85192822e-02],
         [-4.56339629e-17,  5.95131279e-19, -1.29961283e-04],
         [ 3.03509053e-02, -2.12991118e-04,  4.19621317e-19]],

        [[ 5.83890243e-17,  4.54458938e-19, -1.29961283e-04],
         [ 5.19517472e-19,  1.24380178e-16, -1.85192822e-02],
         [-2.12991118e-04, -3.03509053e-02, -1.24394228e-16]],

        [[-8.69176910e-03, -9.44582225e-19, -9.88568179e-19],
         [-9.72498562e-19, -8.69176910e-03, -1.41787196e-16],
         [-1.44676644e-18, -2.06792304e-16,  1.52922918e-02]]],


       [[[-4.12866376e-16, -3.04622052e-18,  2.84431226e-1

In [10]:
dir(A)

['C',
 'Dia',
 'Dijab',
 'Eocc1',
 'Eocc2',
 'Evir1',
 'Evir2',
 'F',
 'FinalEnergy',
 'H1',
 'Hocc1',
 'Hocc2',
 'Hvir1',
 'Hvir2',
 'J1',
 'Jia',
 'Jia1',
 'Jia1mag',
 'Jia2',
 'Jia2mag',
 'Jocc1',
 'Jocc2',
 'Jvir1',
 'Jvir2',
 'K1',
 'Kia',
 'Kia1',
 'Kia1mag',
 'Kia2',
 'Kia2mag',
 'Kocc1',
 'Kocc2',
 'Kvir1',
 'Kvir2',
 'Loc_occ',
 'Loc_vir',
 'MO',
 'PlotMP2Approx',
 'StartEnergy',
 'TestMP2Approx',
 'Triples',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'build_Fae',
 'build_Fme',
 'build_Fmi',
 'build_Wmbej',
 'build_Wmbje',
 'build_Wmnij',
 'build_Zmbij',
 'build_tau',
 'build_tilde_tau',
 'ccsd_corr_e',
 'ccsd_e',
 'compute_corr_energy',
